# Machine Learning for Molecular Simulation

## Clustering of a molecular dynamics trajectory

In this exercise, we will analyze a trajectory of a molecular dynamics simulation of a polypeptide molecule to discover meta-stable states. The peptide is hexamer of alanine amino-acids, which was simulated at a high temperature of $T=500$ K for 20 ns. Every 10 ps, a frame of atomic positions was taken, quenched to $T=0$ K (i.e. geometry optimized), and written to the trajectory file.

The main question that we aim to answer in this exercise is: can we discover some patterns in the many structural configurations that such a flexible molecule samples by grouping the structures based on similarity?

For the grouping, we will use the clustering method *K-Means clustering*.

We simply could supply the atomic positions of each trajectory frame as input to the algorithm to group similar structures, but that would not be a good idea.


$\color{DarkBlue}{\textbf{Question 1}}$
* What could be a limiting disadvantage of directly using the atomic coordinates from the simulation trajectory as input for algorithms that group similar structures?

Instead, we will first compute a list of structural features and store them in a data file. These features will be used to group structures using the clustering algorithm.

Tasks to fulfill:
1. use the [MD-Analysis package](https://userguide.mdanalysis.org/stable/index.html) to import the molecular dynamics trajectory and compute geometric features
1. use the [NGL Viewer package](http://nglviewer.org/nglview/latest/) to view and visually inspect the MD trajectory
1. compute relevant features to characterize the peptide structure
1. use [pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html) to store the features in a dataframe and perform some exploratory data analysis
1. use the [Scikit Learn package](https://scikit-learn.org/stable/index.html) to implement a first version of [K-means clustering](https://scikit-learn.org/stable/modules/clustering.html#k-means)
1. write your own K-means clustering algorithm with an improved distance metric
1. assess the quality of the clustering
1. compute an elbow plot to determine the optimal number of clusters $K$

Additional optional tasks:

8. compute the relative free energy of the clusters
9. compute the transition matrix
10. use the [Scikit Learn package](https://scikit-learn.org/stable/index.html) to perform [K-Nearest Neighbors classification](https://scikit-learn.org/stable/modules/neighbors.html#nearest-neighbors-classification)



In [ ]:
# We need a few packages that we may not have yet installed
# Let's check which packages need to be installed with micromamba/conda install (or pip install):

try:
    import MDAnalysis as mda
except ImportError:
    print("MDAnalysis is not installed.")
    print("Try installing with: micromamba install MDAnalysis -c conda-forge")
try:
    import nglview as nv
except ImportError:
    print("nglview is not installed")
    print("Try installing with: micromamba install nglview -c conda-forge")
try:
    import pandas as pd
except ImportError:
    print("pandas is not installed")
    print("Try installing with: micromamba install pandas -c conda-forge")

In [ ]:
# First load some useful libraries
import MDAnalysis as mda
import nglview as nv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Fixing the random state for reproducibility
np.random.seed(19680801)


## Loading and visualizing the MD trajectory

* load the trajectory in the MD-Analysis "universe",
* we will use for the trajectory file <code>trajectory.xyz</code> provided with this exercise,
* we are not loading a "topology" of the molecule, but use instead the "to_guess" option, so that MD Analysis constructs a list of bonds, angles, and dihedral angels.

In [ ]:
u = mda.Universe("trajectory.xyz",to_guess=['bonds','angles','dihedrals','masses'], dt=10)
u

Use commands such as <code>list(u.atoms)</code>, <code>list(u.angles)</code> and <code>list(u.dihedrals)</code> to see the atom names and lists of angles and dihedrals. use slicing to see only part of the lists:

### Load the trajectory in nglview and visualize the simulation result.
See [this website](https://www.mdanalysis.org/2016/03/14/nglview/) with instructions.

_Note that sometimes the nglview widget does not show the molecule in Jupyter lab. Try then with Jupyter notebook instead_.


In [ ]:
molecule = u.select_atoms('all')
# ======== start your code here =================================


# ======== end your code here ===================================

Note that the alanine hexamer molecule is capped by an acetyl group and an amine group at the two ends.

## Calculation of features, part 1

Compute the following two features:
* the radius of gyration
* the root mean squared displacement (RMSD) with respect to the first frame

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li>use <code>for ts in u.trajectory:</code> to iterate through the trajectory frames.
    <li><code>len(u.trajectory)</code> gives the number of frames
    <li><code>ts.frame</code> gives the frame number 
    <li>see <a href="https://docs.mdanalysis.org/2.0.0/documentation_pages/overview.html">here</a> for an example code on the MD-Analysis website </li>
    <li>for RMSD, see also <a href="https://userguide.mdanalysis.org/stable/examples/analysis/alignment_and_rms/rmsd.html">here</a>
</ul>
</details>

In [ ]:
# Radius of gyration
molecule = u.select_atoms('all')
nframes = len(u.trajectory)
frame = np.zeros(nframes)
radgyr = np.zeros(nframes)
# ======== start your code here ===================================


# ======== end your code here ===================================




Use matplotlib to plot the radius of gyration as a function of frame number

In [ ]:
# Plot radius of gyration versus frame number
plt.figure()
plt.plot(frame,radgyr,"o")
plt.xlabel("frame")
plt.ylabel("radius of gyration")
plt.show()

In [ ]:
# RMSD
# note that the resulting object has a shape of (2000, 3). The last column has the RMSD for each frame.

from MDAnalysis.analysis import rms

R = rms.RMSD(u,  # universe to align
             u,  # reference universe or atomgroup
             select='all',  # group to superimpose and calculate RMSD
             ref_frame=0)  # frame index of the reference
R.run()

Use matplotlib to plot the RMSD as a function of frame number

In [ ]:
# Plot RMSD versus frame number

rmsd = R.results.rmsd[:,2]

plt.figure()
plt.plot(frame,rmsd,"o")
plt.xlabel("frame")
plt.ylabel("rmsd")
plt.show()

$\color{DarkBlue}{\textbf{Question 2}}$
* what do you observe from the radius of gyration and from the RMSD?

Next, let's make a map from these two features by plotting the radius of gyration versus the RMSD. You should already be able to see a little bit of structure or clustering of states.

In [ ]:
# Plot RMSD versus the radius of gyration
# ======== start your code here ===================================



# ======== end your code here ===================================

## Calculation of features, part 2

The RMSD and Radius of Gyration are features that give information on the global structure of the molecule. Next, we will compute a number of dihedral angles, which give more local information on a specific part of the molecule. In particular, we will compute the $\phi$ and $\psi$ angles, which are the backbone dihedral angles on either side of the peptide bond. Not all values of $(\phi, \psi)$ are accessible due to steric hindrance with the amino-acid sidechain, which can be seen in the so-called [Ramachandren plot](https://en.wikipedia.org/wiki/Ramachandran_plot). In the polyalanine hexamer, there are 6 $\phi$ and 6 $\psi$ angles.

Compute the 12 dihedral angles, and store them together with the time, the radius of gyration, and the rmsd in a pandas dataframe.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> Make first a dictionary of the first elements of the feature arrays
    <li> Add the dictionary to a dataframe so that the dictionary items become the column headers
    <li> The relevant dihedral numbers are: 3, 7, 15, 19, 27, 39, 43, 51, 55, 63, 67.
</ul>
</details>


In [ ]:
# make a list of the first row
row = { 'time': u.trajectory.time,
        'radgyr': radgyr[0],
        'rmsd': rmsd[0],           
        'dihedral3': u.dihedrals[3].value(),
        'dihedral7': u.dihedrals[7].value(),
        'dihedral15': u.dihedrals[15].value(),
        'dihedral19': u.dihedrals[19].value(),
        'dihedral27': u.dihedrals[27].value(),
        'dihedral31': u.dihedrals[31].value(),
        'dihedral39': u.dihedrals[39].value(),
        'dihedral43': u.dihedrals[43].value(),
        'dihedral51': u.dihedrals[51].value(),
        'dihedral55': u.dihedrals[55].value(),
        'dihedral63': u.dihedrals[63].value(),
        'dihedral67': u.dihedrals[67].value() 
}
# create the list and append the row
lst = []
lst.append(row)
lst

In [ ]:
# Loop over the trajectory, update a list with each row; at the end append the dictionary to the dataframe
for ts in u.trajectory[1:]:
# ======== start your code here =================================

    
# ======== end your code here ===================================
df_features = pd.DataFrame(lst)
print(df_features)

## Exploratory data analysis (EDA)

It is a good practice to inspect the data before using it for a learning task. When looking at the data, ask such questions as:
* on what ranges do the features vary?
* is the feature range sampled evenly by the data?
* are there outliers?
* are there missing data values?
* are there NaNs in the data?

Use the following commands for a first visual inspection of the dataframe:
<code>df_features.head()</code>, <code>df_features.tail()</code>, <code>df_features.dtypes</code>, <code>df_features.info()</code>, <code>df_features.index</code>, <code>df_features.columns</code>, <code>df_features.describe()</code>, <code>df_features.plot(y="dihedral3")</code>, <code><code>df_features.plot(y="dihedral3",kind="hist")</code>.

### Feature scaling 

Note that the average and variance shown for the features, as seen with <code>df_features.describe()</code>, are rather different for the different features. This can lead to problems when used as such as input for a machine learning algorithm. A commonly used strategy is to scale the features so that they have comparable ranges, typically to the range [0, 1]. Here, we will only consider rsmd and radius of gyration, which span similar ranges, and, separately, the dihedral angles, which also span the same range. We will therefore omit feature scaling for now. But for clustering, which involves computing Eucledian distances in the feature space, using both sets of features at the same time, omitting scaling could give unsatisfactory results.

### Plotting the dihedral angle distributions

Let's have a look at the sampling of the dihedral angles during a simulation (i.e. as a function of time) and also as a distribution (i.e. a histogram):

In [ ]:
# Let's plot on a 2 x 6 grid the phi and psi angles as a function of time for each amino-acid.

dihedrals_start_at = list(df_features.columns).index("dihedral3")
dihedral_names = df_features.columns[dihedrals_start_at:]

fig, axs = plt.subplots(nrows=6, ncols=2, sharey=True, figsize=(12, 16))
axs = axs.flatten()  # don't want to think about i/j
for i, n in enumerate(dihedral_names):
    ax = axs[i]
    ax.scatter(
        df_features.time, df_features[n] , s=6, alpha=0.4, color=f"C{i}"
    )  # add some color
    if i > 8:
        ax.set_xlabel("time / ps")
    ax.set_ylabel(n)
plt.tight_layout()
plt.show()

Note the rather narrow ranges of dihedral angles are being sampled in some plots. Already from these plots it becomes clear that the molecule adopts distinct configurations. If we would assume that the accessible dihedral angle values are independent, then we may even estimate how many meta-stable structures (or clusters) the molecule samples. Let's make and examine two more plots to make this even more clear, before proceeding with actually finding the clusters.

* First, create again a 2 x 6 grid of figures, but now showing histograms of the sampled dihedral angles
* Second, create a 2 x 3 grid of figures showing $\phi$ versus $\psi$.

In [ ]:
# Let's plot on a 2 x 6 grid the histograms of the phi and psi angles for each amino-acid.

fig, axs = plt.subplots(nrows=6, ncols=2, sharey=True, figsize=(12, 16))
axs = axs.flatten()  # don't want to think about i/j
for i, n in enumerate(dihedral_names):
    ax = axs[i]
    ax.hist(df_features[n], range=(-180,180), bins=50, density=True, histtype="bar")
    ax.set(xlim=(-180, 180), xticks=np.arange(-180, 180, 45))
    if i % 2 == 0:
        ax.set_ylabel("population")
    ax.set_xlabel(n)
plt.tight_layout()
plt.show()



In [ ]:
# Plot on a 2 x 3 grid phi versus psi for each amino-acid.
# NB compare the results with an alanine Ramachandran plot seen for example on wikipedia

# ======== start your code here ===================================



# ======== end your code here ===================================

## K-Means clustering

We will now cluster the sampled structures using the K-means clustering algorithm, using different sets of features. We will start by using only the two global features, radius of gyration and RMSD. Next, we investigate the clustering result using the dihedral angles, and thirdly, we will use all features.

K-Means clustering is actually a rather simple algorithm, which is not difficult to implement. As this implementation is rather insightful to get a good grasp of clustering, we will do this below. 

However, to get an idea of what we should get as a result from K-Means clustering, we will first use an implementation from the [Scikit-learn](https://scikit-learn.org/stable/index.html) library. Scikit-learn is a very useful machine learning package for python, containing various algorithms for regression, classification, dimensional reduction and clustering. Since the different classes, functions, and attributes in Scikit-learn have very similar structures and layout, it is very easy to switch from one machine learning method to another.

To prepare our input data, copy first the RMSD and the radius of gyration data into a numpy array of shape (2, nsamples). Then cluster the data into k = 4 clusters.
See [here](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans) for the documentation of the Scikit-Learn K-Means class.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> First create the object by calling the class <code>sklearn.cluster.KMeans()</code> with the number of desired clusters.
    <li> Next call the function <code>kmeans.fit()</code>
</ul>
</details>

Note, K-Means clustering is an unsupervised learning method which has the number of clusters, $K$, as a hyperparameter. What would be a good value for $K$? This is arbitrary, problem dependent, and often not obvious in advance. We may have to try several values of $K$, and judge what is the best (or at least a reasonable) value based on chemical or physics insight. There are also some data-based tools to estimate $K$, for example from a so-called elbow plot. This will be discussed later.



In [ ]:
# Import the sklearn clustering lib
from sklearn.cluster import KMeans

# convert the RMSD and radius of gyration data in a separate feature array with shape(2, nsamples)
features = df_features.loc[:, ("radgyr","rmsd")].values
# print(features)

# set the number of clusters (= K) and cluster the data
# ======== start your code here =================================


# ======== end your code here ===================================


### Clustering results 
Let's have a look at the results. We have now created the clusters, and each cluster is defined by its central position (called "centroid") and its members, which are the data points assigned to the cluster.

* print the variable <code>kmeans.cluster_centers_</code> to see the centroid positions
* print the variable <code>kmeans.labels_</code> to see which data point is assigned to which cluster.
* calculate the numbers of members of each cluster and store them in an np.array <code>nmembers[nclusters]</code>.


In [ ]:
# ======== start your code here =================================



# ======== end your code here ===================================

### Plot the results
Next, make a plot of the results using a scatter plot of RMSD versus radius of gyration, coloured by cluster number. Draw also the cluster centroid positions in the plot.

In [ ]:
plt.figure(dpi=100)
# ======== start your code here ===================================



# ======== end your code here ===================================
plt.xlabel("radius of gyration")
plt.ylabel("rmsd")
plt.show()

To initialize the clustering algorithm, it starts by assigning random cluster centroid positions. The algorithm is therefore not deterministic (unless you initialize the random number seed ([random_state](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#examples-using-sklearn-cluster-kmeans)) to a fix integer).

$\color{DarkBlue}{\textbf{Question 3}}$
* what do you observe in the results when you run the same K-Means clustering several times?

Now, run the above clustering and plotting steps for several different values of the number of clusters.

$\color{DarkBlue}{\textbf{Question 4}}$
* can you say anything about a good value for $K$?

## Assessment of the clustering

How can we evaluate how well the clustering algorithm performed? If we could associate an error estimate to the result, then we could compare this error for different values of $K$ and find the optimal number of clusters as the one that minimizes the error.

Unfortunately, there is no straightforward way to estimate the performance of the clustering task. At least not in a manner similar to estimating the accuracy of a regression task. The reason is that we are dealing with unsupervised learning and we do not have "ground-truth" labels to compare our result with. Several strategies have been developed to assess the performance of the clustering, but most involve a comparison with labeled data points for which the cluster assignment are thus known. See [here](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation) for a description of such strategies.

Here, we will make use of a "score" estimator that is implemented in the K-Means module of Scikit-learn. The [description of](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) the score() function is rather cryptic: "Opposite of the value of X on the K-means objective". 

The objective of the algorithm is to find cluster centers that minimize the sum of (squared) distances between  each center and its members. In the hypothetical case that the member points are all the same for each cluster, then these distances are all zero. So, also if the number of clusters is equal to the number of data points, this objective is zero. If, instead, the clusters are very diffuse and spread out, these distances will be large. The "opposite" of the objective means here the negative.

Let's compute this negative score as a function of $K$ and construct a so-called elbow plot.

In [ ]:
# make an elbow plot
loss = []
nclusters = range(2, 15)
# ======== start your code here =================================



# ======== end your code here ===================================

plt.plot(nclusters, loss, "o-")
plt.xlabel("Cluster Number")
plt.ylabel("Loss")
plt.title("Elbow Plot")
plt.show() 

In [ ]:
# make an elbow plot
loss = []
nclusters = range(2, 15)
# ======== start your code here =================================



# ======== end your code here ===================================

plt.plot(nclusters, loss, "o-")
plt.xlabel("Cluster Number")
plt.ylabel("Loss")
plt.title("Elbow Plot")
plt.show()

The reason that it is called "elbow plot" is because, ideally, the curve has the shape of an elbow, and the optimal number of clusters is supposedly near the kink in the graph. With some squinting of the eyes, possibly two kinks can be discerned.

Let's see if we can discover some physical ground for these numbers of clusters by comparing the structures in the different clusters.

The cluster center represents the average of the data points (i.e. the feature values) assigned to the cluster, but, unfortunately, the cluster center does not represent a physical molecular structure. Associating these average feature values of the centroid to a molecular structure of atom positions would require a generative model, which will be discussed later in the course.

But we can examine and compare the molecular structures that belong to the same cluster. So let's try this as follows:
* recompute the K-Means clusters for the $K$ at an elbow curve kink
* create a new universe trajectory that contains only the structures of one cluster. See [here](https://userguide.mdanalysis.org/stable/trajectories/slicing_trajectories.html) for documentation on slicing/filtering the trajectory using an boolean array (which you can create using the kmeans labels array).
* visualize the trajectory of selected cluster members
* do the above for each cluster
* do the above for both kinks in the elbow plot



In [ ]:
from MDAnalysis.analysis import align

# First recompute the K-Means clusters at the first kink in the elbow plot
# ======== start your code here =================================



# ======== end your code here ===================================


# convert the labels array into a boolean array that selects the members of the desired cluster
# and use it to filter u.trajectory
# ======== start your code here =================================


# ======== end your code here ===================================


# write the selected structures of this cluster to file  
filename = f"cluster_{selected_cluster}.xyz"
u.select_atoms("all").write(filename, frames=u.trajectory[bools])
# read the selected structures into a new universe
u_cluster = mda.Universe(filename,to_guess=['bonds','angles','dihedrals','masses'],dt=10)

# let's align the structure by removing translation and rotation motions
ref = mda.Universe(filename,to_guess=['bonds','angles','dihedrals','masses'],dt=10)
aligner = align.AlignTraj(u_cluster, ref, select='all', in_memory=True).run()


# visualise the cluster members as in the beginning of this exercise
# ======== start your code here =================================


# ======== end your code here ===================================




$\color{DarkBlue}{\textbf{Question 5}}$

Do the structures in the clusters resemble each other when choosing the number of clusters at the first kink in the elbow plot or are there still rather different structures in a cluster? What about when choosing $K$ at the second kink?

## Implementing the K-Means algorithm

Next, we will write up the K-Means algorithm ourselves. In addition to being an insightful exercise, this also allows us to modify the Euclidian distance metric that is used to measure the distance of each data point to the cluster centroids during the cluster assignment. Since the dihedral angle is a periodic function (note for example that the "distance" between an angle of -170 degrees and an angle of 170 degrees is 20 degrees and not 340 degrees), we will use so-called periodic boundary conditions in the distance metric.



Let's first write a function <code>distance(x1, x2, lbox)</code> that returns the distance between two vectors $x1$ and $x2$, after applying periodic boundary conditions:

$\vec{d} := \vec{x}_1 - \vec{x}_2$

$\vec{d} := \vec{d} - L\cdot u_r(\vec{d}/L)$

$r := |\vec{d}|$

in which $u_r()$ is the roundoff function <code>np.round()</code> and $L$ is the periodic "box" length, which is 360 for dihedral angles. 

Note that we can try our algorithm first on the radius of gyration and rmsd without a problem, because their variance is much smaller than 360.

Complete below the function to compute the PBC corrected distance.

In [ ]:
# set the periodic box length to 360
lbox = 360

# write the function for the distance calculation
# ======== start your code here =================================



# ======== end your code here ===================================


Use the following calls of the function to test it. It should print <code>28.284</code> for this example.

In [ ]:
# To test the function, let's compute the Euclidian distance between two structures 
# that are described by two dihedral angles.

structure1 = np.array([10.0, 170])
structure2 = np.array([30.0,-170])
print(f"Distance = {distance(structure1, structure2, lbox):6.3f}")

The K-Means algorithm has the following steps:
1. Initialize the centroid position for each cluster. We will do this by setting it equal to a randomly chosen data point.
1. loop over the data points and set a label for each point equal to the cluster number which has the closest centroid.
1. re-compute the centroid positions as the mean of the positions of its cluster members
1. goto step 2 and repeat steps 2-4 until convergence


$\color{DarkBlue}{\textbf{Question 6}}$
* what could be a useful measure to determine convergence?

Complete below the program for the K-Means algorithm.

In [ ]:
# import useful libraries
import random

# initialize and define some variables
ncluster = 8 
ndata, nfeature = features.shape
centers = np.zeros((ncluster, nfeature))
labels = [0] * ndata
# we will compare the numbers of cluster members with those from the previous step to check convergence
nmembers_prev = np.zeros(ncluster)


# assign the cluster center positions to the positions of randomly chosen data points
# ======== start your code here =================================



# ======== end your code here ===================================


# let's plot the data in gray and add the evolution of the centers in color in the loop 
plt.figure(dpi=100)
plt.scatter(features[:,0],features[:,1],c="LightGray")

icycle = 0
maxcycle = 100
change = (nmembers_prev + 9999.0)
while (change*change).sum() > 0  and icycle < maxcycle:
    # loop over data points and assign each to closest center, i.e. set data point label to cluster number
    for idata in range(ndata):
        rmin = 9999999.
# ======== start your code here =================================



# ======== end your code here ===================================



# recompute the cluster centers by adding up the member positions and then divide by the number of members
    nmembers = np.zeros(ncluster)
# ======== start your code here =================================



# ======== end your code here ===================================
# compute the change in the numbers of members in this cycle
    change = (nmembers - nmembers_prev)
    nmembers_prev = nmembers
    icycle = icycle + 1
    
# print some feedback and plot the new cluster centers
    print(f" cycle: {icycle:3d}, number of cluster members: {nmembers}")
    plt.scatter(centers[:,0],centers[:,1],c=range(ncluster), cmap='rainbow')


plt.show()

# plot a second plot with the cluster members coloured
plt.figure(dpi=100)
plt.scatter(features[:,0],features[:,1],c=labels, cmap='rainbow')
plt.scatter(centers[:,0],centers[:,1],marker="x",c="Black")
plt.show()

### Analysis

Plot again the cluster centers, the labels array, and the numbers of members, and compare your clustering result with that from Scikit-Learn. Is it comparable?
 

In [ ]:
# ======== start your code here =================================



# ======== end your code here =================================


## K-Means clustering using the dihedral angles

Now, we are ready to cluster the structures using the dihedral angles as input features.
Let's fill our feature array with dihedral angle data from the data frame and run your K-Means clustering implementation using the PBC-distance metric.

In [ ]:
# convert data into features
features = df_features.loc[:,dihedral_names].values
time = df_features.time.values
features.shape

In [ ]:
# run your K-means clustering program using the new features array
# ======== start your code here =================================






# ======== end your code here ===================================



In [ ]:
# ======== start your code here =================================



# ======== end your code here ===================================
centers.shape

In [ ]:
# Plot on a 2 x 3 grid phi versus psi for each amino-acid.
# NB compare the results with a alanine Ramachandran plot seen for example on wikipedia
# ======== start your code here =================================



# ======== end your code here ===================================

In [ ]:
# visuale the cluster structures
# Repeat this code block to visually inspect each cluster (labeled by "iclus")
iclus = 0
print(f"  Cluster: {iclus},  Number of members: {int(nmembers[iclus])}")
bools = [labels[j] == iclus for j in range(len(labels))]
filename = f"cluster_{iclus}.xyz"
u.select_atoms("all").write(filename, frames=u.trajectory[bools])
u_cluster = mda.Universe(filename,to_guess=['bonds','angles','dihedrals','masses'],dt=10)
ref = mda.Universe(filename,to_guess=['bonds','angles','dihedrals','masses'],dt=10)
aligner = align.AlignTraj(u_cluster, ref, select='all', in_memory=True).run()
    
w = nv.show_mdanalysis(u_cluster)
w



$\color{DarkBlue}{\textbf{Question 7}}$
* Wat are your findings on (1) what is the better performing feature set (and why?) and (2) how many meta-stable states does this molecule have (i.e. what should $K$ be)?

## Conclusions

This is the end of the AI4Chem Tutorial on Clustering.

You have learned how to cluster data using the unsupervised learning method K-Means clustering.
You were also introduced to the Scikit-Learn library for machine learning and to the MD-Analysis and NGL-Viewer packages to read, analyze and visualize molecular structures and simulation trajectories. Finally, we used pandas and matplotlib for storing, manipulating, and plotting data sets.

In case you would like to still go a bit further with this assignment topic, here are a few ideas to tryout:
* Finding the meta-stable states that a (bio-)molecule samples is the first step in a modeling approach called Markov state modeling. The idea is that the number of members in a cluster is related to the probability for the molecule to visit that state, which is related to the free energy of the state. In particular, $\Delta G(i)=-k_B T \ln[n_i / N]$, in which $n_i$ is the number of structures in cluster $i$, and $N$ is the total number of structures. With this formula you can now calculate the relative free energies of the clusters.
* In Markov state modeling, the transition matrix gives the rates of the transition from state $i$ to state $j$. These can be estimates from the labels array, simply by counting how often label $i$ is followed by label $j$, which gives the number of transitions between each pair of states. Dividing these numbers by the total time of the simulation gives an estimator of the transition matrix. By projecting the states as a 2D graph, in which the nodes are the states and the edges represent the transitions, one obtains mechanistic insight on the pathways that the molecule can follow to transition from one conformation to another.
* To estimate how different or similar the clustering results are between the two sets of features that we used, one can of course compare the nmembers arrays. A better estimator of the discrepancy would be to loop over the labels array, and count how many data points have been assigned differently between the two approaches. Note that this requires that the clusters have the same label numbers. This can be enforced by assigning the initial centroid positions by hand to specific data points, for example the closest data points to the centroids in a previous run.
* At the end of this exercise, we now have in fact labeled data. This can be used for supervised learning. K-Nearest Neighbors (KNN) is a clustering method that uses supervised learning. It can be used to assign new data points to previously computed or assigned clusters by measuring the distance to the $K$ (for example 3) closest data points, and then assigns a label based on the labels of these closest neighbors. An interesting exercise to try out KNN clustering could be to use the previous data points from the simulation of the peptide at $T=500$ Kelvin, which were labeled with K-Means clustering, to now cluster a new trajectory from a simulation of the peptide at $T=300$ Kelvin (included in this exercise) using KNN. See [here](https://stackabuse.com/k-nearest-neighbors-algorithm-in-python-and-scikit-learn/) for an instructive explanation for using KNN using Scikit-Learn.